# Load project
Based on [fastmachinelearning/hls4ml-tutorial/part7b_deployment.ipynb](https://github.com/fastmachinelearning/hls4ml-tutorial/blob/main/part7b_deployment.ipynb), [Tanawin1701d/vitisUnifiedTutorial/part8b_testOnHw.ipynbxt](https://github.com/Tanawin1701d/vitisUnifiedTutorial/blob/main/part8b_testOnHw.ipynb) and earlier experimentation. For syntax, see driverclass in directory and [pynq-package documentation](https://pynq.readthedocs.io/en/v2.5.1/pynq_package/pynq.overlay.html).

In [11]:
# import the library
from axi_master_driver import NeuralNetworkOverlay
import numpy as np

In [12]:
# Load input from .npy file
x_test = np.load('x_test.npy')
y_test = np.load('y_test.npy')

#input_array  = x_test.astype(np.float16) 
#output_array = np.zeros(y_test.shape, dtype=np.float16)

# Allocate physically contiguous memory for input and output
#input_buffer = allocate(shape=input_array.shape, dtype=np.float16)
#output_buffer = allocate(shape=output_array.shape, dtype=np.float16)

# check input shape
#print(f"input array shape {input_array.shape}")
#print(f"output array shape {output_array.shape}")

In [13]:
# create the overlay object
overlay = NeuralNetworkOverlay(bitfile_name="2025.2/system.bit", x_shape=x_test.shape, y_shape=y_test.shape, dtype=x_test.dtype)
help(overlay)
#overlay?

Help on NeuralNetworkOverlay in module axi_master_driver:

<axi_master_driver.NeuralNetworkOverlay object>
    Default documentation for overlay 2025.2/system.bit. The following
    attributes are available on this overlay:
    
    IP Blocks
    ----------
    axi_dma_0            : pynq.lib.dma.DMA
    axi_intc_0           : pynq.overlay.DefaultIP
    myproject_axi_master_1 : axi_master_driver.HLS4ML_IP
    zynq_ultra_ps_e_0    : pynq.overlay.DefaultIP
    
    Hierarchies
    -----------
    None
    
    Interrupts
    ----------
    None
    
    GPIO Outputs
    ------------
    None
    
    Memories
    ------------
    PSDDR                : Memory



In [25]:
x = x_test
y_hardware = overlay.predict(x, debug=False, profile=True, encode=np.float32, decode=np.float32)


input gmem_in0_ptr_linput will be set to addr: 0x3c600000 with elements: 2656000
output gmem_out0_ptr_layer5_out will be set to addr: 0x3a900000 with elements: 830000
amount of queries will be set to: 166000 at address: 0x28
prepare your interrupt
global interrupt enable register
enable gie successful
ap_done interrupt enable register
enable ap_done interrupt successful
ap_done register clear
clear ap_done interrupt successful
----------------------
starting the accelerator
Accelerator execution time:                       1.427 ms
accelerator has finished
Processed elements: 166000, Execution time: 0.0014 seconds, Performance rate: 116309784.80 inferences/second


In [27]:
y_hardware

(PynqBuffer([[0.5       , 0.42773438, 0.04101562, 0.04882812, 0.10449219],
             [0.07421875, 0.6669922 , 0.00585938, 0.23046875, 0.13085938],
             [0.18945312, 0.07128906, 0.7998047 , 0.        , 0.        ],
             ...,
             [0.02441406, 0.09277344, 0.04492188, 0.7998047 , 0.16210938],
             [0.06640625, 0.15820312, 1.        , 0.        , 0.00292969],
             [0.10742188, 0.09472656, 0.08398438, 0.5       , 0.24316406]],
            dtype=float32),
 0.0014272230000642594,
 116309784.80064152)

In [31]:
y_hardware[0]

PynqBuffer([[0.5       , 0.42773438, 0.04101562, 0.04882812, 0.10449219],
            [0.07421875, 0.6669922 , 0.00585938, 0.23046875, 0.13085938],
            [0.18945312, 0.07128906, 0.7998047 , 0.        , 0.        ],
            ...,
            [0.02441406, 0.09277344, 0.04492188, 0.7998047 , 0.16210938],
            [0.06640625, 0.15820312, 1.        , 0.        , 0.00292969],
            [0.10742188, 0.09472656, 0.08398438, 0.5       , 0.24316406]],
           dtype=float32)

In [22]:
np.argmax(y_hardware)

17

In [23]:
y_test

array([[0., 1., 0., 0., 0.],
       [0., 1., 0., 0., 0.],
       [0., 0., 1., 0., 0.],
       ...,
       [0., 1., 0., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0.]], dtype=float32)

In [32]:
np.save("y_hardware.npy",y_hardware[0])

....

In [19]:
# copy data to input buffer
np.copyto(input_buffer, input_array)
input_buffer.flush()

In [20]:
# get the ip and initialize the system
ip = overlay.myproject_axi_master_1  # Replace with your IP instance name
ip.set_input (0, input_buffer)
ip.set_output(0, output_buffer)
#ip.set_amt_query(input_array.shape[0]) # object has no attribute 'REG_ADDR_AMT_QUERY'
ip.prepare_intr()

input gmem_in0_ptr_linput will be set to addr: 0x3b900000 with elements: 2656000
output gmem_out0_ptr_layer5_out will be set to addr: 0x3a700000 with elements: 830000
prepare your interrupt
global interrupt enable register
enable gie successful
ap_done interrupt enable register
enable ap_done interrupt successful
ap_done register clear
clear ap_done interrupt successful
----------------------


In [21]:
async def wait_for_acc():
    print("starting the accelerator")
    ip.ctrl_start()
    print("waiting for the accelerator to finish")
    await my_interrupt.wait()
    print("accelerator has finished")


# Inference

In [22]:
#### get event loop from asyncio
loop = asyncio.get_event_loop()

In [23]:
task = loop.create_task(wait_for_acc())
loop.run_until_complete(task)

starting the accelerator
waiting for the accelerator to finish
accelerator has finished


In [24]:
output_buffer.invalidate()

In [25]:
print(input_buffer)
print(output_buffer)

[[-0.11950312  0.40616292 -1.0405861  ...  0.4086576  -1.0199556
  -0.18016747]
 [ 0.30887297  0.2271202  -1.1560557  ...  1.8674464  -1.2320029
  -1.1949688 ]
 [-1.2655021   0.66753286  1.4087453  ...  0.58246976  1.3092331
   1.7571807 ]
 ...
 [ 1.1456269  -0.46692902 -0.36909324 ... -1.1835415   0.05067312
  -0.6875682 ]
 [-0.1047317   0.3005048   1.3434744  ... -0.52088165  1.3800093
   0.23497856]
 [-0.91237694  0.729601    0.17986116 ... -0.29436678  0.09422757
   0.5578699 ]]
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 ...
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]


In [13]:
# convert it to numpy array
print("we got output shape:", output_buffer.shape)
outNp = np.array(output_buffer)

we got output shape: (166000, 5)


In [14]:
# save it to .npy file
np.save("out_hw.npy", outNp)